# Shadow Price Prediction - Example Usage

This notebook demonstrates how to use the shadow price prediction pipeline with customizable parameters.

In [1]:
from pbase.config.ray import init_ray

import shadow_price_prediction
extra_modules = [shadow_price_prediction]
init_ray(address='ray://10.8.0.36:10001', extra_modules=extra_modules)

import pandas as pd
from shadow_price_prediction import (
    ShadowPricePipeline,
    PredictionConfig,
    FeatureConfig,
    ModelConfig,
    ModelSpec,
    EnsembleConfig,
    TrainingConfig,
    ThresholdConfig,
    AnomalyDetectionConfig,
    DataPathConfig,
    calculate_metrics,
    print_metrics_report,
    # Model classes - now imported for direct use
    XGBClassifier,
    XGBRegressor,
    LogisticRegression,
    ElasticNet,
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    LinearRegression,
    StackingModel,
    HyperparameterTuner
)

2025-11-19 08:02:43,294	INFO client_builder.py:242 -- Passing the following kwargs to ray.init() on the server: log_to_driver
2025-11-19 08:02:43,679	INFO packaging.py:588 -- Creating a file package for local module '/home/xyz/workspace/research-spice-shadow-price-pred/src/shadow_price_prediction'.
2025-11-19 08:02:43,682	INFO packaging.py:380 -- Pushing file package 'gcs://_ray_pkg_db78e000dfdd7318.zip' (0.19MiB) to Ray cluster...
2025-11-19 08:02:43,686	INFO packaging.py:393 -- Successfully pushed file package 'gcs://_ray_pkg_db78e000dfdd7318.zip'.
SIGTERM handler is not set because current thread is not the main thread.


## 1. Configure Parameters

Set all configurable parameters for the prediction pipeline.

In [2]:
# ============================================================================
# Date and Market Parameters
# ============================================================================
period_type = 'f0'
class_type = 'onpeak'
market_round = 1

# ============================================================================
# Data Path Templates
# ============================================================================
density_path_template = (
    '/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/density/'
    'auction_month={auction_month}/market_month={market_month}/'
    'market_round={market_round}/outage_date={outage_date}'
)

constraint_path_template = (
    '/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/constraint_info/'
    'auction_month={auction_month}/market_round={market_round}/'
    'period_type={period_type}/class_type={class_type}'
)

# ============================================================================
# ============================================================================
# Feature Configuration
# ============================================================================
# Define common feature set for both classification and regression
common_features = [
    # Raw Density Shape
    '110', '105', '100', '95', '90',
    # Probability Mass Intervals (Diffs)
    '105_diff', '100_diff', '95_diff', '90_diff',
    '85_diff', '80_diff', '70_diff', '60_diff',
    # Engineered Risk Features (Full Tail Integration)
    'prob_overload', 'risk_ratio', 'curvature_100',
    'prob_exceed_110', 'prob_exceed_105', 'prob_exceed_100', 'prob_exceed_95', 'prob_exceed_90',
    'log_density_100',
    # Interaction Features
    'interaction_risk_overload', 'interaction_curvature_exceed'
]

feature_config = FeatureConfig(
    step1_features=common_features,
    step2_features=common_features
)

# ============================================================================
# Ensemble Configuration - Simplified with ModelConfig
# ============================================================================
# NEW APPROACH: Use actual model classes (XGBClassifier, LogisticRegression, etc.)
# All parameters are passed dynamically via ModelConfig(params={...})
# No need for separate config classes for each model type!

ensemble_config = EnsembleConfig(
    # Default classifier ensemble: XGBoost (50%) + LogisticRegression (50%)
    default_classifiers=[
        ModelSpec(
            model_class=XGBClassifier,  # Pass the actual class
            config=ModelConfig(params={  # All parameters in params dict
                'n_estimators': 500,
                'min_child_weight': 10,
                'n_jobs': -1,
                'max_depth': 6,
                'learning_rate': 0.1,
                'verbosity': 0,
                'eval_metric': 'logloss',
            }),
            weight=0.6
        ),
        ModelSpec(
            model_class=LogisticRegression,
            config=ModelConfig(params={
                'C': 1.0,
                'max_iter': 1000,
                'class_weight': 'balanced',
                'n_jobs': -1,
                'solver': 'lbfgs',
            }),
            weight=0.4
        )
    ],
    
    # Branch-specific classifier ensemble: XGBoost (50%) + LogisticRegression (50%)
    branch_classifiers=[
        ModelSpec(
            model_class=XGBClassifier,
            config=ModelConfig(params={
                'n_estimators': 500,
                'max_depth': 6,
                'learning_rate': 0.1,
                'min_child_weight': 5,
                'n_jobs': 1,
                'verbosity': 0,
                'eval_metric': 'logloss',
            }),
            weight=0.6
        ),
        ModelSpec(
            model_class=LogisticRegression,
            config=ModelConfig(params={
                'C': 1.0,
                'max_iter': 1000,
                'class_weight': 'balanced',
                'n_jobs': 1,
                'solver': 'lbfgs',
            }),
            weight=0.4
        )
    ],
    
    # Default regressor ensemble: XGBoost (50%) + ElasticNet (50%)
    default_regressors=[
        ModelSpec(
            model_class=XGBRegressor,
            config=ModelConfig(params={
                'n_estimators': 500,
                'min_child_weight': 2,
                'n_jobs': -1,
                'max_depth': 6,
                'learning_rate': 0.1,
                'verbosity': 0,
                'objective': 'reg:squarederror',
            }),
            weight=0.6
        ),
        ModelSpec(
            model_class=ElasticNet,
            config=ModelConfig(params={
                'alpha': 1.0,
                'l1_ratio': 0.5,
                'max_iter': 1000,
                'fit_intercept': True,
            }),
            weight=0.4
        )
    ],
    
    # Branch-specific regressor ensemble: XGBoost (50%) + ElasticNet (50%)
    branch_regressors=[
        ModelSpec(
            model_class=XGBRegressor,
            config=ModelConfig(params={
                'n_estimators': 500,
                'max_depth': 6,
                'learning_rate': 0.1,
                'min_child_weight': 1,
                'n_jobs': 1,
                'verbosity': 0,
                'objective': 'reg:squarederror',
            }),
            weight=0.6
        ),
        ModelSpec(
            model_class=ElasticNet,
            config=ModelConfig(params={
                'alpha': 1.0,
                'l1_ratio': 0.5,
                'max_iter': 1000,
                'fit_intercept': True,
            }),
            weight=0.4
        )
    ]
)

# ============================================================================
# Advanced Examples: Easy to Mix Any Models!
# ============================================================================
# Example 1: XGBoost + RandomForest Classifier Ensemble
# ------------------------------------------------------
# ensemble_config = EnsembleConfig(
#     default_classifiers=[
#         ModelSpec(
#             model_class=XGBClassifier,
#             config=ModelConfig(params={
#                 'n_estimators': 200,
#                 'max_depth': 4,
#                 'subsample': 0.8,  # Any XGBoost parameter!
#                 'colsample_bytree': 0.8,
#                 'random_state': 42
#             }),
#             weight=0.6
#         ),
#         ModelSpec(
#             model_class=RandomForestClassifier,
#             config=ModelConfig(params={
#                 'n_estimators': 100,
#                 'max_depth': 10,
#                 'criterion': 'entropy',  # Any RandomForest parameter!
#                 'max_features': 'sqrt',
#                 'random_state': 42
#             }),
#             weight=0.4
#         )
#     ]
# )

# Example 2: Three-Model Ensemble
# ---------------------------------
# ensemble_config = EnsembleConfig(
#     default_classifiers=[
#         ModelSpec(model_class=XGBClassifier, 
#                   config=ModelConfig(params={'n_estimators': 200}), weight=0.5),
#         ModelSpec(model_class=LogisticRegression, 
#                   config=ModelConfig(params={'C': 1.0, 'max_iter': 1000}), weight=0.4),
#         ModelSpec(model_class=GradientBoostingClassifier, 
#                   config=ModelConfig(params={'n_estimators': 100, 'learning_rate': 0.1}), weight=0.2)
#     ]
# )

# ============================================================================
# Training Configuration
# ============================================================================
training_config = TrainingConfig(
    min_samples_for_branch_model=10,
    min_binding_samples_for_regression=1,
    train_months_lookback=12  # 12 months of training data
)

# ============================================================================
# Threshold Optimization Configuration
# ============================================================================
threshold_config = ThresholdConfig(
    threshold_range_start=0.01,
    threshold_range_end=0.99,
    threshold_range_steps=99,
    threshold_beta=2  # F-beta score beta parameter
)

# ============================================================================
# Anomaly Detection Configuration
# ============================================================================
anomaly_config = AnomalyDetectionConfig(
    enabled=True,
    k_multiplier=3.0,  # IQR multiplier for anomaly threshold
    min_samples_for_stats=10,
    flow_feature='100'  # Feature to use for anomaly detection
)

# ============================================================================
# Additional Parameters
# ============================================================================
run_at_day = 10
outage_freq = '3D'  # Outage date frequency

### Example 3: Stacking Ensemble

Use the `StackingModel` to train a meta-model on top of base models.

In [ ]:
# Define base models
xgb_base = XGBClassifier(n_estimators=100, max_depth=4, eval_metric='logloss', n_jobs=1)
lr_base = LogisticRegression(max_iter=1000, n_jobs=1)

# Define Stacking Ensemble Configuration
# ensemble_config = EnsembleConfig(
#     default_classifiers=[
#         ModelSpec(
#             model_class=StackingModel,
#             config=ModelConfig(params={
#                 'base_models': [xgb_base, lr_base],
#                 'meta_model': LogisticRegression(),
#                 'use_proba': True
#             }),
#             weight=1.0  # Stacking model takes full weight
#         )
#     ]
# )

## 2. Create Pipeline Configuration

Combine all parameters into a single configuration object.

**Important: Parallel Per-Period Training Behavior**

When using multiple test periods, the pipeline will use **Ray for parallel processing**:
1. For each test period (processed in PARALLEL), calculate the training period based on that period's `auction_month`
2. Train NEW models on that period's training data (each period on separate worker)
3. Make predictions for that period
4. Combine results across all periods

**Example with 2 test periods (processed in parallel)**:
- Period 1 (2025-10): Trains on 2024-10 to 2025-09, predicts for 2025-10 [Worker 1]
- Period 2 (2025-11): Trains on 2024-11 to 2025-10, predicts for 2025-11 [Worker 2]

**Performance**: ~2x speedup for 2 periods, ~3x for 3 periods (on machines with enough cores)

**Two options available:**
- **Option 1**: Single test period (backward compatible)
- **Option 2**: Multiple test periods (batch processing with parallel per-period training) - uncomment to use

In [3]:
# Create path configuration
path_config = DataPathConfig(
    density_path_template=density_path_template,
    constraint_path_template=constraint_path_template
)

# ============================================================================
# Option 1: Single Test Period (Backward Compatible)
# ============================================================================
# config = PredictionConfig(
#     # Date parameters - single period
#     test_auction_month=test_auction_month,
#     test_market_month=test_market_month,
    
#     # Market parameters
#     period_type=period_type,
#     class_type=class_type,
#     market_round=market_round,
    
#     # Path configuration
#     paths=path_config,
    
#     # Feature configuration
#     features=feature_config,
    
#     # Ensemble configuration
#     models=ensemble_config,
    
#     # Training configuration
#     training=training_config,
    
#     # Threshold optimization
#     threshold=threshold_config,
    
#     # Anomaly detection
#     anomaly_detection=anomaly_config,
    
#     # Data loading parameters
#     run_at_day=run_at_day,
#     outage_freq=outage_freq
# )

# print("✓ Single-period configuration created successfully")

# ============================================================================
# Option 2: Multiple Test Periods (Batch Processing)
# ============================================================================
# Uncomment to process multiple periods in a single run:

test_periods_list = [
    (pd.Timestamp('2025-06'), pd.Timestamp('2025-06')),
    (pd.Timestamp('2025-07'), pd.Timestamp('2025-07')),
    (pd.Timestamp('2025-08'), pd.Timestamp('2025-08')),
    (pd.Timestamp('2025-09'), pd.Timestamp('2025-09')),
    (pd.Timestamp('2025-10'), pd.Timestamp('2025-10')),
]

config = PredictionConfig(
    # Date parameters - multiple periods
    test_periods=test_periods_list,
    
    # Market parameters
    period_type=period_type,
    class_type=class_type,
    market_round=market_round,
    
    # Path configuration
    paths=path_config,
    
    # Feature configuration
    features=feature_config,
    
    # Ensemble configuration
    models=ensemble_config,
    
    # Training configuration
    training=training_config,
    
    # Threshold optimization
    threshold=threshold_config,
    
    # Anomaly detection
    anomaly_detection=anomaly_config,
    
    # Data loading parameters
    run_at_day=run_at_day,
    outage_freq=outage_freq
)

print(f"✓ Batch configuration created successfully for {len(test_periods_list)} periods")
print(f"\n📊 Ensemble Configuration:")
print(f"\n  Classification Ensembles:")
print(f"    Default Classifiers: {len(ensemble_config.default_classifiers)} models")
for i, spec in enumerate(ensemble_config.default_classifiers, 1):
    print(f"      {i}. {spec.model_class.__name__}: weight={spec.weight:.2f}")
print(f"    Branch Classifiers: {len(ensemble_config.branch_classifiers)} models")
for i, spec in enumerate(ensemble_config.branch_classifiers, 1):
    print(f"      {i}. {spec.model_class.__name__}: weight={spec.weight:.2f}")

print(f"\n  Regression Ensembles:")
print(f"    Default Regressors: {len(ensemble_config.default_regressors)} models")
for i, spec in enumerate(ensemble_config.default_regressors, 1):
    print(f"      {i}. {spec.model_class.__name__}: weight={spec.weight:.2f}")
print(f"    Branch Regressors: {len(ensemble_config.branch_regressors)} models")
for i, spec in enumerate(ensemble_config.branch_regressors, 1):
    print(f"      {i}. {spec.model_class.__name__}: weight={spec.weight:.2f}")

✓ Batch configuration created successfully for 5 periods

📊 Ensemble Configuration:

  Classification Ensembles:
    Default Classifiers: 2 models
      1. XGBClassifier: weight=0.60
      2. LogisticRegression: weight=0.40
    Branch Classifiers: 2 models
      1. XGBClassifier: weight=0.60
      2. LogisticRegression: weight=0.40

  Regression Ensembles:
    Default Regressors: 2 models
      1. XGBRegressor: weight=0.60
      2. ElasticNet: weight=0.40
    Branch Regressors: 2 models
      1. XGBRegressor: weight=0.60
      2. ElasticNet: weight=0.40


## 3. Run the Pipeline

Execute the complete prediction pipeline.

**Parallel Processing (Default for Multiple Periods)**:
- When you have multiple test periods, the pipeline uses Ray to train models in parallel
- Each period is processed independently on separate workers
- Significant speedup: ~3x faster for 3 periods on machines with 3+ cores
- Set `use_parallel=False` to disable parallel processing (for debugging)

**Controlling Parallel Workers with `n_jobs`**:
- `n_jobs=0` (default): Auto-determine based on available CPUs
- `n_jobs=3`: Use exactly 3 workers (useful for 3 test periods)
- `n_jobs=-1`: Use all available CPUs (maximum parallelization)
- `n_jobs=1`: Effectively sequential processing

In [4]:
# Create pipeline
pipeline = ShadowPricePipeline(config)

# ============================================================================
# Option 1: Auto-determine workers (default) - Recommended
# ============================================================================
# Ray will automatically determine the optimal number of workers based on:
# - Number of test periods
# - Available CPU cores
# This is the recommended setting for most use cases
results_per_outage, final_results, metrics = pipeline.run(
    verbose=True, 
    use_parallel=True,
    n_jobs=6  # Auto-determine (default)
)

SHADOW PRICE PREDICTION PIPELINE - PARALLEL PER-PERIOD TRAINING MODE
Test Periods: 5
  1. Auction: 2025-06, Market: 2025-06
  2. Auction: 2025-07, Market: 2025-07
  3. Auction: 2025-08, Market: 2025-08
  4. Auction: 2025-09, Market: 2025-09
  5. Auction: 2025-10, Market: 2025-10
Class Type: onpeak
Period Type: f0

🚀 Using Ray parallel processing for 5 periods
   Workers: 6

[PARALLEL PROCESSING: Training and Predicting for All Periods]


(PoolActor pid=28024, ip=10.42.7.70) 25-11-19 08:02:50 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=28024, ip=10.42.7.70) 25-11-19 08:02:50 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=28024, ip=10.42.7.70) 25-11-19 08:02:50 EST | INFO    | pbase.config.base    | load_global_config  | #435 | Loading config from local cached json file...
(PoolActor pid=28024, ip=10.42.7.70) 25-11-19 08:02:50 EST | INFO    | pbase.config.base    | __init__            | #44  | Init global config...
(PoolActor pid=28349, ip=10.42.4.173) 25-11-19 08:02:50 EST | INFO    | dotenv.main          | _get_stream         | #61  | python-dotenv could not find configuration file .env.
(PoolActor pid=28349, ip=10.42.4.173) 25-11-19 08:02:50 EST | WARNING | pbase.utils.utils    | load_env            | #478 | .env could not be found...
(PoolActor pid=28349, ip=10.42

(PoolActor pid=28024, ip=10.42.7.70) 
(PoolActor pid=28024, ip=10.42.7.70) ================================================================================
(PoolActor pid=28024, ip=10.42.7.70) [Processing Period 2]
(PoolActor pid=28024, ip=10.42.7.70)   Auction Month: 2025-07
(PoolActor pid=28024, ip=10.42.7.70)   Market Month:  2025-07
(PoolActor pid=28024, ip=10.42.7.70) ================================================================================
(PoolActor pid=28024, ip=10.42.7.70) 
(PoolActor pid=28024, ip=10.42.7.70) [STEP 1: Training Period Calculation]
(PoolActor pid=28024, ip=10.42.7.70)   Training Range: 2024-07 to 2025-07
(PoolActor pid=28024, ip=10.42.7.70) 
(PoolActor pid=28024, ip=10.42.7.70) [STEP 2: Loading Training Data]
(PoolActor pid=28024, ip=10.42.7.70) 
(PoolActor pid=28024, ip=10.42.7.70) [Loading Training Data]
(PoolActor pid=28024, ip=10.42.7.70) --------------------------------------------------------------------------------
(PoolActor pid=28024, ip=10.42.7

  0%|          | 0/5 [00:00<?, ?it/s]


[COMBINING RESULTS ACROSS ALL PERIODS]

✓ Results Combined
  Total samples (per-outage): 1,359,474
  Total unique constraints (monthly): 130,534
  Date range: 2025-06-01 to 2025-10-31

[ANALYZING COMBINED RESULTS]

[MONTHLY AGGREGATED RESULTS]

Test Period: Monthly Level
  Total samples: 130,534

[Shadow Price Prediction Performance]
  MAE:  $348.51
  RMSE: $1880.27
  MAPE: 719.28%

[Classification Performance]
  Precision: 0.325
  Recall:    0.328
  F1-Score:  0.326
  Accuracy:  0.885

[Binding Classification Details]
  Correctly classified: 115,581 / 130,534 (88.54%)

  True Positives (correctly identified binding): 3,621
  False Positives (incorrectly predicted as binding): 7,524
  True Negatives (correctly identified non-binding): 111,960
  False Negatives (missed binding constraints): 7,429

[Binding Rate Analysis]
  Actual binding rate: 8.47% (11,050 samples)
  Predicted binding rate: 8.54% (11,145 samples)

[PER-OUTAGE SUMMARY]

Total outage dates: 53

Average metrics across ou

## 4. Analyze Results

The results are already analyzed during the pipeline run. You can access:
- `results_per_outage`: Per-outage-date level predictions
- `final_results`: Monthly aggregated predictions
- `metrics`: Comprehensive metrics for both levels

In [5]:
# Access metrics
monthly_metrics = metrics['monthly']
per_outage_metrics = metrics['per_outage']

print("\nMonthly Metrics:")
print(f"  MAE: ${monthly_metrics['mae']:.2f}")
print(f"  RMSE: ${monthly_metrics['rmse']:.2f}")
print(f"  F1-Score: {monthly_metrics['f1_score']:.3f}")
print(f"  Precision: {monthly_metrics['precision']:.3f}")
print(f"  Recall: {monthly_metrics['recall']:.3f}")


Monthly Metrics:
  MAE: $348.51
  RMSE: $1880.27
  F1-Score: 0.326
  Precision: 0.325
  Recall: 0.328


(raylet, ip=10.42.4.173) Spilled 10863 MiB, 3 objects, write throughput 1777 MiB/s. Set RAY_verbose_spill_logs=0 to disable this message.
(raylet, ip=10.42.9.131) Spilled 7754 MiB, 2 objects, write throughput 1778 MiB/s. Set RAY_verbose_spill_logs=0 to disable this message.


In [ ]:
# Access metrics
monthly_metrics = metrics['monthly']
per_outage_metrics = metrics['per_outage']

print("\nMonthly Metrics:")
print(f"  MAE: ${monthly_metrics['mae']:.2f}")
print(f"  RMSE: ${monthly_metrics['rmse']:.2f}")
print(f"  F1-Score: {monthly_metrics['f1_score']:.3f}")
print(f"  Precision: {monthly_metrics['precision']:.3f}")
print(f"  Recall: {monthly_metrics['recall']:.3f}")


Monthly Metrics:
  MAE: $342.90
  RMSE: $1893.19
  F1-Score: 0.346
  Precision: 0.295
  Recall: 0.418


## 5. Analyze Specific Outage Dates (Optional)

Calculate metrics for individual outage dates.

In [7]:
# Example: Analyze a specific outage date
specific_date = '2025-10-01'

if specific_date in per_outage_metrics:
    outage_metrics = per_outage_metrics[specific_date]
    print_metrics_report(
        outage_metrics,
        title=f"Metrics for Outage Date: {specific_date}",
        level="outage"
    )
else:
    print(f"No metrics found for {specific_date}")


[Metrics for Outage Date: 2025-10-01]

Test Period: Outage Level
  Total samples: 12,646

[Shadow Price Prediction Performance]
  MAE:  $85.99
  RMSE: $479.37
  MAPE: 525.78%

[Classification Performance]
  Precision: 0.172
  Recall:    0.248
  F1-Score:  0.203
  Accuracy:  0.918

[Binding Classification Details]
  Correctly classified: 11,608 / 12,646 (91.79%)

  True Positives (correctly identified binding): 132
  False Positives (incorrectly predicted as binding): 637
  True Negatives (correctly identified non-binding): 11,476
  False Negatives (missed binding constraints): 401

[Binding Rate Analysis]
  Actual binding rate: 4.21% (533 samples)
  Predicted binding rate: 6.08% (769 samples)


## 6. Export Results (Optional)

Save results to files for further analysis.

In [ ]:
# Save results to CSV
# results_per_outage.to_csv('results_per_outage.csv')
# final_results.to_csv('final_results_monthly.csv')
# print("✓ Results saved to CSV files")

## 7. Top Predictions Analysis

Analyze the top predicted shadow prices.

In [ ]:
# Top 20 predictions by predicted shadow price
top_20_predictions = (
    final_results
    .sort_values('predicted_shadow_price', ascending=False)
    .drop_duplicates(subset=['branch_name'])
    .head(20)
)

print("\nTop 20 Predicted Shadow Prices:")
print(top_20_predictions[[
    'branch_name', 
    'binding_probability', 
    'predicted_shadow_price', 
    'actual_shadow_price', 
    'error',
    'model_used'
]].to_string())

In [ ]:
# Top 20 actual shadow prices for comparison
top_20_actual = (
    final_results
    .sort_values('actual_shadow_price', ascending=False)
    .drop_duplicates(subset=['branch_name'])
    .head(20)
)

print("\nTop 20 Actual Shadow Prices:")
print(top_20_actual[[
    'branch_name', 
    'binding_probability', 
    'predicted_shadow_price', 
    'actual_shadow_price', 
    'error',
    'model_used'
]].to_string())